# 01 — WAHIS Data Audit

## Machine Learning-Based Early Warning System for Poultry Disease Outbreaks in Nigeria

This notebook documents the initial audit of the World Organisation for Animal Health (WOAH) WAHIS quantitative poultry disease surveillance data.

The purpose of this stage is to:

1. Load the raw WAHIS dataset.
2. Inspect the dataset structure and dimensions.
3. Examine the available disease categories.
4. Assess temporal and geographic coverage.
5. Identify missing and unavailable values.
6. Examine the structure of outbreak-count variables.
7. Document important data-quality limitations before cleaning and modelling.

### Important methodological principle

A missing or unavailable outbreak report must **not** automatically be interpreted as zero outbreaks.

Therefore, this audit distinguishes between:

- explicitly reported outbreak counts;
- missing/unavailable values; and
- values that can legitimately be interpreted as zero.

This distinction is important because incorrectly converting missing surveillance reports into zero outbreaks could introduce false negative observations into the machine-learning dataset.

## 1. Import Required Libraries

The following libraries are used for data loading, inspection and basic exploratory auditing.

The notebook uses `pathlib.Path` for file paths so that the project remains portable across environments.

In [5]:
# ============================================================
# CELL 3 — IMPORT LIBRARIES AND DEFINE PROJECT PATHS
# ============================================================

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# Determine the project root.
#
# When this notebook is executed from the notebooks/ directory,
# Path.cwd() points to:
#
#     poultry-disease-early-warning-nigeria/notebooks
#
# Therefore, we move one level upward to reach the project root.
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# ------------------------------------------------------------
# Define the location of the raw WAHIS quantitative dataset.
# ------------------------------------------------------------

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Quantitative data 2026-10-01.csv"
)


# ------------------------------------------------------------
# Display the paths so that we can verify that the notebook
# is pointing to the correct project location.
# ------------------------------------------------------------

print("Current notebook directory:")
print(CURRENT_DIR)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nWAHIS dataset path:")
print(DATA_PATH)

print("\nDoes the dataset exist?")
print(DATA_PATH.exists())

Current notebook directory:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\notebooks

Project root:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria

WAHIS dataset path:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\raw\Quantitative data 2026-10-01.csv

Does the dataset exist?
True


In [6]:
# ============================================================
# CELL 4 — LOAD THE RAW WAHIS DATASET
# ============================================================

# Load the raw WAHIS quantitative data.
#
# At this stage we deliberately load the dataset without
# performing cleaning or transformation. The purpose of this
# notebook is to audit the original data first.

df = pd.read_csv(DATA_PATH)


# ------------------------------------------------------------
# Display the basic dimensions of the raw dataset.
# ------------------------------------------------------------

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Rows: 1,180
Columns: 19


## 2. Initial Dataset Inspection

Before modifying the dataset, we inspect its structure, column names, data types and first observations.

No cleaning is performed in this section. The objective is to understand what was actually supplied in the raw WAHIS extract.

In [8]:
# ============================================================
# CELL 6 — PREVIEW THE FIRST OBSERVATIONS
# ============================================================

# Display the first five rows of the raw dataset.
# This provides an initial view of the records and their
# structure before any transformations are applied.

df.head()

,Year,Semester,World region,Country,Administrative Division,Disease,Serotype/Subtype/Genotype,Animal Category,Event_id,Species,Outbreak_id,New outbreaks,Susceptible,Measuring units,Cases,Killed and disposed of,Slaughtered,Deaths,Vaccinated
0,2005,Jul-Dec 2005,Africa,Nigeria,Nigeria,Fowl pox (-2005),NaN,Domestic,-,Birds,-,-,90,Animal,60,0,0,0,0
1,2005,Jul-Dec 2005,Africa,Nigeria,Nigeria,Fowl typhoid,NaN,Domestic,-,Birds,-,-,1000,Animal,800,0,0,25,0
2,2005,Jul-Dec 2005,Africa,Nigeria,Nigeria,Infectious bursal disease (Gumboro disease),NaN,Domestic,-,Birds,-,-,200,Animal,10,0,0,0,0
3,2005,Jul-Dec 2005,Africa,Nigeria,Nigeria,Newcastle disease virus (Inf. with),NaN,Domestic,-,Birds,-,-,22100,Animal,3527,0,800,2076,0
4,2005,Jan-Jun 2005,Africa,Nigeria,Nigeria,Newcastle disease virus (Inf. with),NaN,Domestic,-,Birds,-,-,8100,Animal,3430,0,0,912,13000


In [7]:
# ============================================================
# CELL 7 — INSPECT THE DATASET STRUCTURE
# ============================================================

# Display the DataFrame structure, including:
#
# - column names
# - number of non-null observations
# - detected data types
#
# This is useful for identifying columns that may require
# cleaning or type conversion during the next stage.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1180 entries, 0 to 1179
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Year                       1180 non-null   int64 
 1   Semester                   1180 non-null   object
 2   World region               1180 non-null   object
 3   Country                    1180 non-null   object
 4   Administrative Division    1179 non-null   object
 5   Disease                    1180 non-null   object
 6   Serotype/Subtype/Genotype  260 non-null    object
 7   Animal Category            1180 non-null   object
 8   Event_id                   1180 non-null   object
 9   Species                    735 non-null    object
 10  Outbreak_id                1180 non-null   object
 11  New outbreaks              1180 non-null   object
 12  Susceptible                1180 non-null   object
 13  Measuring units            1180 non-null   object
 14  Cases   

In [9]:
# ============================================================
# CELL 8 — DISPLAY COLUMN NAMES
# ============================================================

# Display all column names exactly as they appear in the
# original WAHIS extract.

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

01. Year
02. Semester
03. World region
04. Country
05. Administrative Division
06. Disease
07. Serotype/Subtype/Genotype
08. Animal Category
09. Event_id
10. Species
11. Outbreak_id
12. New outbreaks
13. Susceptible
14. Measuring units
15. Cases
16. Killed and disposed of
17. Slaughtered
18. Deaths
19. Vaccinated


In [10]:
# ============================================================
# CELL 9 — CHECK MISSING VALUES
# ============================================================

# Calculate the number of missing values in every column.
#
# This is an audit operation only. We are NOT replacing,
# dropping or converting missing values at this stage.

missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing_summary

Serotype/Subtype/Genotype    920
Species                      445
Administrative Division        1
World region                   0
Semester                       0
Year                           0
Country                        0
Disease                        0
Animal Category                0
Event_id                       0
Outbreak_id                    0
New outbreaks                  0
Susceptible                    0
Measuring units                0
Cases                          0
Killed and disposed of         0
Slaughtered                    0
Deaths                         0
Vaccinated                     0
dtype: int64

In [11]:
# ============================================================
# CELL 10 — CALCULATE MISSING-VALUE PERCENTAGES
# ============================================================

# Calculate the percentage of records that are missing
# for each column.

missing_percentage = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

missing_percentage

Serotype/Subtype/Genotype    77.966102
Species                      37.711864
Administrative Division       0.084746
World region                  0.000000
Semester                      0.000000
Year                          0.000000
Country                       0.000000
Disease                       0.000000
Animal Category               0.000000
Event_id                      0.000000
Outbreak_id                   0.000000
New outbreaks                 0.000000
Susceptible                   0.000000
Measuring units               0.000000
Cases                         0.000000
Killed and disposed of        0.000000
Slaughtered                   0.000000
Deaths                        0.000000
Vaccinated                    0.000000
dtype: float64

## 3. Disease Coverage

The WAHIS dataset contains records covering multiple poultry diseases.

We inspect the disease field to determine which disease categories are represented in the raw extract.

The disease distribution will inform the later decision about the modelling target and whether a disease-specific modelling strategy is appropriate.

In [12]:
# ============================================================
# CELL 12 — INSPECT DISEASE CATEGORIES
# ============================================================

# Count the number of records associated with each disease.
#
# dropna=False ensures that missing disease values are also
# visible rather than silently excluded.

disease_counts = (
    df["Disease"]
      .value_counts(dropna=False)
)

disease_counts

Disease
Newcastle disease virus (Inf. with)            610
Infectious bursal disease (Gumboro disease)    324
Fowl typhoid                                   192
Avian infectious bronchitis                     51
Fowl cholera (-2011)                             2
Fowl pox (-2005)                                 1
Name: count, dtype: int64

In [13]:
# Display the number of unique disease categories.

print(
    f"Number of unique disease values: "
    f"{df['Disease'].nunique(dropna=True)}"
)

Number of unique disease values: 6


In [14]:
# ============================================================
# CELL 14 — INSPECT SEMESTER REPORTING
# ============================================================

# Count observations by semester.
#
# Missing semester values are retained in the output so that
# incomplete reporting can be identified during the audit.

semester_counts = (
    df["Semester"]
      .value_counts(dropna=False)
)

semester_counts

Semester
Jul-Dec 2021    134
Jul-Dec 2025    113
Jan-Jun 2022    106
Jul-Dec 2022    100
Jan-Jun 2021     91
Jan-Jun 2025     76
Jan-Jun 2023     72
Jan-Jun 2024     71
Jul-Dec 2023     68
Jul-Dec 2024     66
Jul-Dec 2020     45
Jan-Jun 2020     28
Jan-Jun 2008     23
Jul-Dec 2008     19
Jul-Dec 2017     13
Jan-Jun 2012     12
Jan-Jun 2009     11
Jul-Dec 2012     11
Jan-Jun 2011     11
Jul-Dec 2018      9
Jan-Jun 2019      9
Jan-Jun 2013      9
Jul-Dec 2009      9
Jan-Jun 2010      8
Jul-Dec 2010      7
Jan-Jun 2015      7
Jul-Dec 2013      7
Jul-Dec 2019      6
Jul-Dec 2014      6
Jul-Dec 2015      5
Jan-Jun 2017      5
Jan-Jun 2018      5
Jan-Jun 2014      4
Jul-Dec 2005      4
Jul-Dec 2011      4
Jan-Jun 2016      2
Jul-Dec 2016      2
Jan-Jun 2005      1
Jul-Dec 2007      1
Name: count, dtype: int64

In [15]:
# ============================================================
# WAHIS DATA AUDIT — MISSINGNESS AND UNAVAILABLE VALUES
# ============================================================

# The raw WAHIS dataset contains two important forms of
# missing or unavailable information:
#
# 1. NaN
#    These are values that pandas recognises as missing.
#
# 2. "-"
#    This is a textual marker appearing in the original WAHIS
#    extract. It should NOT automatically be interpreted as 0.
#
# This distinction is critical for this research because:
#
#     missing/unavailable outbreak information
#                 ≠
#             zero outbreaks
#
# Therefore, this section only AUDITS these values.
# It does not modify the raw dataset.


# ------------------------------------------------------------
# 1. Count pandas NaN values
# ------------------------------------------------------------

nan_counts = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)


# ------------------------------------------------------------
# 2. Count textual "-" markers
#
# We temporarily convert the DataFrame to pandas' nullable
# string representation so that we can safely identify "-"
# without changing the original df.
# ------------------------------------------------------------

dash_counts = (
    df.astype("string")
      .eq("-")
      .sum()
      .sort_values(ascending=False)
)


# ------------------------------------------------------------
# 3. Combine both forms of missing/unavailable information
# into one audit table.
# ------------------------------------------------------------

missing_audit = pd.DataFrame({
    "column": df.columns,
    "NaN_count": df.isna().sum().values,
    "dash_count": (
        df.astype("string")
          .eq("-")
          .sum()
          .values
    ),
})

missing_audit["total_missing_or_unavailable"] = (
    missing_audit["NaN_count"]
    + missing_audit["dash_count"]
)

missing_audit = (
    missing_audit
    .sort_values(
        "total_missing_or_unavailable",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Display the audit table
# ------------------------------------------------------------

print("Missing and unavailable values by column:")
display(missing_audit)


# ------------------------------------------------------------
# 5. Define the main quantitative surveillance variables.
#
# These variables will be especially important during the
# subsequent data-cleaning and modelling stages.
# ------------------------------------------------------------

surveillance_columns = [
    "New outbreaks",
    "Susceptible",
    "Cases",
    "Killed and disposed of",
    "Slaughtered",
    "Deaths",
    "Vaccinated",
]


# ------------------------------------------------------------
# 6. Inspect the raw data types of the surveillance variables.
#
# We do this BEFORE numeric conversion so that the original
# representation of "-" remains visible.
# ------------------------------------------------------------

print("\nRaw surveillance-variable data types:")
display(df[surveillance_columns].dtypes)


# ------------------------------------------------------------
# 7. Count "-" values specifically in the surveillance fields.
# ------------------------------------------------------------

surveillance_dash_counts = (
    df[surveillance_columns]
      .astype("string")
      .eq("-")
      .sum()
      .sort_values(ascending=False)
)

print("\nUnavailable '-' values in surveillance variables:")
display(surveillance_dash_counts)


# ------------------------------------------------------------
# 8. Display sample records containing unavailable outbreak
# information.
#
# This helps us visually verify how "-" appears alongside
# other reported disease-impact variables.
# ------------------------------------------------------------

unavailable_outbreaks = df[
    df["New outbreaks"].astype("string").eq("-")
]

print("\nSample records where 'New outbreaks' is unavailable:")
display(
    unavailable_outbreaks[
        [
            "Year",
            "Semester",
            "Country",
            "Administrative Division",
            "Disease",
            "New outbreaks",
            "Cases",
            "Deaths",
            "Slaughtered",
            "Vaccinated",
        ]
    ].head(10)
)

Missing and unavailable values by column:


,column,NaN_count,dash_count,total_missing_or_unavailable
0,Outbreak_id,0,1180,1180
1,Event_id,0,1180,1180
2,Serotype/Subtype/Genotype,920,0,920
3,New outbreaks,0,735,735
4,Vaccinated,0,512,512
5,Slaughtered,0,491,491
6,Susceptible,0,468,468
7,Killed and disposed of,0,466,466
8,Cases,0,459,459
9,Deaths,0,456,456



Raw surveillance-variable data types:


New outbreaks             object
Susceptible               object
Cases                     object
Killed and disposed of    object
Slaughtered               object
Deaths                    object
Vaccinated                object
dtype: object


Unavailable '-' values in surveillance variables:


New outbreaks             735
Vaccinated                512
Slaughtered               491
Susceptible               468
Killed and disposed of    466
Cases                     459
Deaths                    456
dtype: Int64


Sample records where 'New outbreaks' is unavailable:


,Year,Semester,Country,Administrative Division,Disease,New outbreaks,Cases,Deaths,Slaughtered,Vaccinated
0,2005,Jul-Dec 2005,Nigeria,Nigeria,Fowl pox (-2005),-,60,0,0,0
1,2005,Jul-Dec 2005,Nigeria,Nigeria,Fowl typhoid,-,800,25,0,0
2,2005,Jul-Dec 2005,Nigeria,Nigeria,Infectious bursal disease (Gumboro disease),-,10,0,0,0
3,2005,Jul-Dec 2005,Nigeria,Nigeria,Newcastle disease virus (Inf. with),-,3527,2076,800,0
4,2005,Jan-Jun 2005,Nigeria,Nigeria,Newcastle disease virus (Inf. with),-,3430,912,0,13000
5,2007,Jul-Dec 2007,Nigeria,Gombe,Infectious bursal disease (Gumboro disease),-,-,25,0,-
6,2008,Jan-Jun 2008,Nigeria,Adamawa,Newcastle disease virus (Inf. with),-,-,32,-,-
7,2008,Jan-Jun 2008,Nigeria,Bauchi,Infectious bursal disease (Gumboro disease),-,969,545,55,-
8,2008,Jan-Jun 2008,Nigeria,Bauchi,Newcastle disease virus (Inf. with),-,922,297,-,-
9,2008,Jan-Jun 2008,Nigeria,Delta,Fowl typhoid,-,50,20,-,-


In [16]:
# ============================================================
# WAHIS DATA AUDIT — GEOGRAPHIC AND TEMPORAL COVERAGE
# ============================================================

# This section examines the geographic and temporal structure
# of the raw WAHIS surveillance data.
#
# No cleaning or transformation is performed here.


# ------------------------------------------------------------
# 1. Geographic coverage — country
# ------------------------------------------------------------

country_counts = (
    df["Country"]
      .value_counts(dropna=False)
)

print("Country coverage:")
display(country_counts)


# ------------------------------------------------------------
# 2. Geographic coverage — world region
# ------------------------------------------------------------

region_counts = (
    df["World region"]
      .value_counts(dropna=False)
)

print("\nWorld-region coverage:")
display(region_counts)


# ------------------------------------------------------------
# 3. Geographic coverage — administrative division
# ------------------------------------------------------------

administrative_division_counts = (
    df["Administrative Division"]
      .value_counts(dropna=False)
)

print("\nAdministrative-division coverage:")
display(administrative_division_counts)


# ------------------------------------------------------------
# 4. Temporal coverage — year
# ------------------------------------------------------------

print("\nTemporal coverage:")

print(f"Minimum year: {df['Year'].min()}")
print(f"Maximum year: {df['Year'].max()}")
print(f"Number of unique years: {df['Year'].nunique()}")


# ------------------------------------------------------------
# 5. Semester values
# ------------------------------------------------------------

semester_values = (
    df["Semester"]
      .dropna()
      .unique()
)

print("\nSemester values:")

for value in sorted(semester_values):
    print(f"- {value}")


# ------------------------------------------------------------
# 6. Year-semester record distribution
# ------------------------------------------------------------

year_semester_counts = (
    df.groupby(
        ["Year", "Semester"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["Year", "Semester"]
    )
)

print("\nRecords by year and semester:")
display(year_semester_counts)

Country coverage:


Country
Nigeria    1180
Name: count, dtype: int64


World-region coverage:


World region
Africa    1180
Name: count, dtype: int64


Administrative-division coverage:


Administrative Division
Osun                         70
Jigawa                       66
Kaduna                       65
Plateau                      57
Ekiti                        55
Kwara                        52
Gombe                        50
Kogi                         46
Niger                        45
Benue                        43
Akwa Ibom                    42
Bayelsa                      38
Rivers                       38
Bauchi                       37
Enugu                        36
Oyo                          31
Ondo                         30
Ogun                         29
Abia                         29
Sokoto                       29
Zamfara                      28
Edo                          27
Borno                        26
Nassarawa                    23
Katsina                      22
Federal Capital Territory    22
Kano                         21
Anambra                      18
Kebbi                        17
Delta                        16
Cross River     


Temporal coverage:
Minimum year: 2005
Maximum year: 2025
Number of unique years: 20

Semester values:
- Jan-Jun 2005
- Jan-Jun 2008
- Jan-Jun 2009
- Jan-Jun 2010
- Jan-Jun 2011
- Jan-Jun 2012
- Jan-Jun 2013
- Jan-Jun 2014
- Jan-Jun 2015
- Jan-Jun 2016
- Jan-Jun 2017
- Jan-Jun 2018
- Jan-Jun 2019
- Jan-Jun 2020
- Jan-Jun 2021
- Jan-Jun 2022
- Jan-Jun 2023
- Jan-Jun 2024
- Jan-Jun 2025
- Jul-Dec 2005
- Jul-Dec 2007
- Jul-Dec 2008
- Jul-Dec 2009
- Jul-Dec 2010
- Jul-Dec 2011
- Jul-Dec 2012
- Jul-Dec 2013
- Jul-Dec 2014
- Jul-Dec 2015
- Jul-Dec 2016
- Jul-Dec 2017
- Jul-Dec 2018
- Jul-Dec 2019
- Jul-Dec 2020
- Jul-Dec 2021
- Jul-Dec 2022
- Jul-Dec 2023
- Jul-Dec 2024
- Jul-Dec 2025

Records by year and semester:


,Year,Semester,record_count
0,2005,Jan-Jun 2005,1
1,2005,Jul-Dec 2005,4
2,2007,Jul-Dec 2007,1
3,2008,Jan-Jun 2008,23
4,2008,Jul-Dec 2008,19
5,2009,Jan-Jun 2009,11
6,2009,Jul-Dec 2009,9
7,2010,Jan-Jun 2010,8
8,2010,Jul-Dec 2010,7
9,2011,Jan-Jun 2011,11


In [17]:
# ============================================================
# WAHIS DATA AUDIT — DISEASE COVERAGE
# ============================================================

# Examine the distribution of disease categories represented
# in the raw WAHIS surveillance dataset.
#
# This is important because the project will later determine
# which disease is most suitable for focused early-warning
# modelling.


# ------------------------------------------------------------
# 1. Count records by disease
# ------------------------------------------------------------

disease_summary = (
    df.groupby(
        "Disease",
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        "record_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Disease distribution:")
display(disease_summary)


# ------------------------------------------------------------
# 2. Number of unique disease categories
# ------------------------------------------------------------

number_of_diseases = df["Disease"].nunique(
    dropna=True
)

print(
    f"\nNumber of unique disease categories: "
    f"{number_of_diseases}"
)

Disease distribution:


,Disease,record_count
0,Newcastle disease virus (Inf. with),610
1,Infectious bursal disease (Gumboro disease),324
2,Fowl typhoid,192
3,Avian infectious bronchitis,51
4,Fowl cholera (-2011),2
5,Fowl pox (-2005),1



Number of unique disease categories: 6


In [18]:
# ============================================================
# WAHIS DATA AUDIT — FINAL SUMMARY
# ============================================================

# Create a concise summary of the main audit findings.
#
# This summary records the characteristics of the raw dataset
# before any cleaning or modelling decisions are made.


print("=" * 60)
print("WAHIS RAW DATA AUDIT SUMMARY")
print("=" * 60)

print(f"Total records: {len(df):,}")
print(f"Total variables: {df.shape[1]:,}")

print(
    f"Year range: "
    f"{df['Year'].min()}–{df['Year'].max()}"
)

print(
    f"Unique diseases: "
    f"{df['Disease'].nunique(dropna=True)}"
)

print(
    f"Unique administrative divisions: "
    f"{df['Administrative Division'].nunique(dropna=True)}"
)

print(
    f"Missing administrative divisions: "
    f"{df['Administrative Division'].isna().sum()}"
)

print(
    f"Missing species values: "
    f"{df['Species'].isna().sum()}"
)

print(
    f"Missing serotype/subtype/genotype values: "
    f"{df['Serotype/Subtype/Genotype'].isna().sum()}"
)

print(
    f"Records with unavailable 'New outbreaks' values: "
    f"{df['New outbreaks'].astype('string').eq('-').sum()}"
)

print("=" * 60)

print("\nKEY METHODOLOGICAL RULE:")
print(
    "Unavailable/missing outbreak information must not "
    "automatically be interpreted as zero outbreaks."
)

WAHIS RAW DATA AUDIT SUMMARY
Total records: 1,180
Total variables: 19
Year range: 2005–2025
Unique diseases: 6
Unique administrative divisions: 39
Missing administrative divisions: 1
Missing species values: 445
Missing serotype/subtype/genotype values: 920
Records with unavailable 'New outbreaks' values: 735

KEY METHODOLOGICAL RULE:
Unavailable/missing outbreak information must not automatically be interpreted as zero outbreaks.


# Initial Audit Conclusion

The raw WAHIS quantitative dataset contains 1,180 records and 19 variables covering temporal, geographic, disease and animal-surveillance information.

The audit identifies both conventional missing values (`NaN`) and textual unavailable markers (`-`). These representations must be distinguished during data cleaning.

In particular, the presence of `-` in `New outbreaks` must not be interpreted as evidence that zero outbreaks occurred. A missing or unavailable surveillance report is not equivalent to a confirmed absence of disease.

The dataset also contains semester-based reporting information, providing the temporal structure required for subsequent feature engineering and chronological model validation.

No values have been imputed, recoded or removed in this notebook. All transformations are intentionally deferred to the WAHIS Data Cleaning stage.

The next stage is:

**02 — WAHIS Data Cleaning**